In [1]:
import os
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from tqdm import tqdm

In [2]:
SEED = 42
MAX_LEN = 256//2
BATCH_SIZE = 16*2#*2# changed for dataparallel
EPOCHS = 10  
MODEL_PATH = "/kaggle/input/deberta-v3-base/transformers/default/1/deberta-v3-base"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# Set seeds
import random
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.cuda.manual_seed_all(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [5]:
# Load and preprocess data
train_path = "/kaggle/input/jigsaw-agile-community-rules/train.csv"
test_path = "/kaggle/input/jigsaw-agile-community-rules/test.csv"
sample_sub_path = "/kaggle/input/jigsaw-agile-community-rules/sample_submission.csv"

In [6]:
df = pd.read_csv(train_path)
df["text"] = df["rule"] + " [SEP] " + df["body"]
df["label"] = df["rule_violation"].astype(float)

In [7]:
def add_data(dataframe):
    ret=[[],[]]
    for i in ['positive_example_1','positive_example_2','negative_example_1','negative_example_2']:
        tmp= (dataframe['rule']+' [SEP] '+ dataframe[i]).tolist()
        ret[0]+= tmp
        ret[1]+= [1]*len(tmp) if 'positive' in i else [0]*len(tmp)
    return ret

In [8]:
test_df = pd.read_csv(test_path)

# Get augmented data from both df and test_df
augmented_train = add_data(df)
augmented_test = add_data(test_df)

# Combine original df with augmented data
augmented_texts =  df.text.tolist()+augmented_train[0] + augmented_test[0]
augmented_labels =  df.label.tolist()+augmented_train[1] + augmented_test[1]

# Create new augmented dataframe
augmented_df = pd.DataFrame({
    'text': augmented_texts,
    'label': augmented_labels
})
print(f'Before:{augmented_df.shape}')
augmented_df = augmented_df.groupby(augmented_df['text'].str.lower(), as_index=False).agg({
    'text': 'first',  # Keep the original case of the first occurrence
    'label': 'mean'   # Take mean of labels
})
print('After:',augmented_df.shape)
augmented_df['rule']= augmented_df.text.apply(lambda x: x.split(' [SEP] ')[0])
augmented_df['body']= augmented_df.text.apply(lambda x: x.split(' [SEP] ')[1])

rule_map= {i:j for j,i in enumerate(augmented_df.rule.str.lower().unique())}
augmented_df['rule_id']= augmented_df.rule.str.lower().map(rule_map)

augmented_df.head()

Before:(10185, 2)
After: (1875, 2)


,text,label,rule,body,rule_id
0,"No Advertising: Spam, referral links, unsolici...",1.0,"No Advertising: Spam, referral links, unsolici...","\n\nIf you have some free time on your hands, ...",0
1,"No Advertising: Spam, referral links, unsolici...",1.0,"No Advertising: Spam, referral links, unsolici...",\n\nplease visit http://www.shifadental.net/te...,0
2,"No Advertising: Spam, referral links, unsolici...",0.0,"No Advertising: Spam, referral links, unsolici...",\n\nSD | [ English Stream 1 Arsenal vs Tottenh...,0
3,"No Advertising: Spam, referral links, unsolici...",0.0,"No Advertising: Spam, referral links, unsolici...",\n**HD** ENG [ 1080P HD Amazing] :- [USTREAM E...,0
4,"No Advertising: Spam, referral links, unsolici...",1.0,"No Advertising: Spam, referral links, unsolici...",\nFree http://forums.airdroid.com/viewtopic.ph...,0


In [9]:
# Load tokenizer locally
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast = False)

In [10]:
class JigsawDataset(Dataset):
    def __init__(self, texts, labels,rule_ids, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.rule_ids = rule_ids

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        enc = self.tokenizer(
            text, padding='max_length', truncation=True, max_length=self.max_len, return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        item['rule_ids']= torch.tensor(self.rule_ids[idx])
        return item

In [11]:
class JigsawModel(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.base = AutoModel.from_pretrained(model_path)
        self.drop = nn.Dropout(0.15)
        self.out = nn.Linear(self.base.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.base(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        return self.out(self.drop(pooled)).squeeze(1)

In [12]:
def train_one_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    for batch in tqdm(loader):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        logits = model(input_ids, mask)
        loss = nn.BCEWithLogitsLoss()(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step()
        if scheduler:
            scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [13]:
def validate(model, loader):
    model.eval()
    preds, targets, rule_ids_list = [], [], []
    total_loss = 0
    criterion = nn.BCEWithLogitsLoss()
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            rule_ids = batch["rule_ids"]  # Assuming this is already on CPU as integers
            
            logits = model(input_ids, mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            
            preds.extend(torch.sigmoid(logits).cpu().numpy())
            targets.extend(labels.cpu().numpy())
            rule_ids_list.extend(rule_ids.cpu().numpy() if torch.is_tensor(rule_ids) else rule_ids)
    
    # Convert to numpy arrays
    preds = np.array(preds)
    targets = np.array(targets)
    rule_ids_array = np.array(rule_ids_list)
    
    # Compute AUC per rule
    unique_rules = np.unique(rule_ids_array)
    rule_aucs = {}
    
    for rule_id in unique_rules:
        rule_mask = rule_ids_array == rule_id
        rule_preds = preds[rule_mask]
        rule_targets = targets[rule_mask]
        
        # Only compute AUC if we have both positive and negative samples for this rule
        if len(np.unique(rule_targets >= 0.5)) > 1:
            rule_auc = roc_auc_score(rule_targets >= 0.5, rule_preds)
            rule_aucs[rule_id] = rule_auc
        else:
            # If only one class present, we can't compute AUC
            rule_aucs[rule_id] = np.nan
    
    # Compute average AUC across rules (excluding NaN values)
    valid_aucs = [auc for auc in rule_aucs.values() if not np.isnan(auc)]
    avg_auc_per_rule = np.mean(valid_aucs) if valid_aucs else 0
    
    val_loss = total_loss / len(loader)
    return avg_auc_per_rule, val_loss, preds

In [14]:
from transformers import get_linear_schedule_with_warmup

In [15]:
unlabelled= pd.read_csv('/kaggle/input/jigsaw-unlabelled-14b/sampled_unlabelled_100k_with_predictions.csv')
unlabelled['text']= unlabelled['rule']+ ' [SEP] '+ unlabelled['body']
unlabelled['rule_id']= unlabelled.rule.str.lower().map(rule_map)
unlabelled.head()

,body,subreddit,similarity_score,rule,pred_yes,pred_no,rule_violation,text,rule_id
0,"Its probably illegal, but Id do it. She may bl...",legaladvice,0.377491,No legal advice: Do not offer or request legal...,0.859664,0.140336,0.859664,No legal advice: Do not offer or request legal...,1
1,Honestly? The jobs are going to decide this. \...,relationships,0.402387,No legal advice: Do not offer or request legal...,0.284576,0.715424,0.284576,No legal advice: Do not offer or request legal...,1
2,"""Uplifting""",UpliftingNews,0.507457,No legal advice: Do not offer or request legal...,0.020023,0.979977,0.020023,No legal advice: Do not offer or request legal...,1
3,And you get this medical info from....your bum?,The_Donald,0.466199,No legal advice: Do not offer or request legal...,0.030215,0.969785,0.030215,No legal advice: Do not offer or request legal...,1
4,got a code message me,hearthstone,0.417036,"No Advertising: Spam, referral links, unsolici...",0.069542,0.930458,0.069542,"No Advertising: Spam, referral links, unsolici...",0


In [16]:
unlabelled_taken= unlabelled.sample(frac=.16)#

In [ ]:
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Create 90/10 split
    train_data, val_data = train_test_split(
        augmented_df, 
        test_size=0.1, 
        stratify=augmented_df["rule"], 
        random_state=SEED
    )
    
    
    # Create datasets
    train_ds = JigsawDataset(
        train_data['text'].tolist()+unlabelled_taken['text'].tolist(), 
        train_data['label'].tolist()+unlabelled_taken['rule_violation'].tolist(), 
        train_data['rule_id'].tolist()+unlabelled_taken['rule_id'].tolist(), 
        tokenizer, MAX_LEN
    )
    
    val_ds = JigsawDataset(
        val_data['text'].tolist(), 
        val_data['label'].tolist(), 
        val_data['rule_id'].tolist(), 
        tokenizer, MAX_LEN
    )
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    
    # Initialize model
    model = JigsawModel(MODEL_PATH).to(DEVICE)
    for name, param in model.named_parameters():
        if name.startswith('base.embedding'):
            param.requires_grad = False
    
    print('Trainable Params:', sum(i.numel() for i in model.parameters() if i.requires_grad))
    # model= nn.DataParallel(model)
    # Setup optimizer and scheduler
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, eps=1e-6)
    total_steps = EPOCHS * len(train_loader)
    warmup_steps = int(0.1 * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )
    
    # Training loop
    best_auc = 0
    for epoch in range(EPOCHS):
        print(f"Epoch {epoch+1}/{EPOCHS}")
        loss = train_one_epoch(model, train_loader, optimizer, scheduler)
        val_auc, val_loss, val_preds = validate(model, val_loader)
        
        print(f"Loss: {loss:.4f}, Val Loss: {val_loss:.4f}, Val AUC: {val_auc:.4f}")
        if val_auc > best_auc:
            best_auc = val_auc
            torch.save(model.state_dict(), "model_single_best.bin")
    
    print(f"Best validation AUC: {best_auc:.4f}")

2025-10-06 00:14:52.192586: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759709692.377089      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759709692.436333      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Trainable Params: 85449985
Epoch 1/10


100%|██████████| 559/559 [07:50<00:00,  1.19it/s]


Loss: 0.4475, Val Loss: 0.4485, Val AUC: 0.8908
Epoch 2/10


 14%|█▍        | 77/559 [01:06<06:57,  1.15it/s]

In [ ]:
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Generate test predictions
    sample = pd.read_csv(sample_sub_path)
    df_test = pd.read_csv(test_path)
    df_test["text"] = df_test["rule"] + " [SEP] " + df_test["body"]
    
    # Load best model
    model = JigsawModel(MODEL_PATH).to(DEVICE)
    model.load_state_dict(torch.load("model_single_best.bin", map_location=DEVICE))
    model.eval()
    
    test_ds = JigsawDataset(df_test['text'].tolist(), [0]*len(df_test), [0]*len(df_test), tokenizer, MAX_LEN)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)
    
    test_preds = []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Generating test predictions"):
            ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            logits = model(ids, mask)
            test_preds.extend(torch.sigmoid(logits).cpu().numpy())

In [ ]:
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    sample["rule_violation"] = test_preds
    sample.to_csv("submission.csv", index=False)
else:
    !touch submission.csv
    
!head -n 4 submission.csv

In [ ]:
#changes not,epochs, folds
#ideas to test-> check if psedo labels matter, at what multiplier, at what weight,at what class balance, at what confidence, extra -ve random sampling

In [ ]:
#ablation
#start                            .8686 3rd epoch .45 val loss
#all pseudo 
#random sample pseudo 2k          .8830 2nd epoch, .43 val loss
#random sample pseudo 4k          .8840 4th epoch, .4305 val loss
#random sample pseudo 8k          .8947 4th epochs, .4197
#random sample pseudo 16k         .8920 3th epoch  .4396
#random sample pseudo 32k         .8913 3th epoch  .4298